In [ ]:
# general imports 
import numpy as np
import os
import matplotlib.pyplot as plt
import math
import pandas as pd
import MDAnalysis as mda
from IPython.display import Video

# Today's Practical

- We will make a step forward toward biological simulations.
- The main idea of today's practical is to see how principal physics behind intermolecular interactions contributes to the known phenomena.

In [ ]:
!ls -l

## What is inside?

### force_field

Now you know what it is. :)

### images

Images related to the practical.

### scripts

Contains `.mdp` files we will check and discuss today.

### structures

Starting atomic configurations of molecules of interest.

### trajectories

Contains long, already conducted simulations for the purposes of today's practical.

### videos

Videos related to the practical.

### week5.ipynb

Jupyter notebook we are using right now.

### Create a folder where we will be working today

In [ ]:
!mkdir playground/

In [ ]:
%cd playground/

## Ions in water

### Role of ions in biology

- Ions are everywhere.
- All biological fluids containt them.
- Multiple roles:
    - Contributing to cellular membrane potential.
    - Participating in neurotransmission.
    - Involved in cell signaling.
    - Maiintaning the osmotic balance.
    - And many more...

### Who knows what is NaCl?

### Preparing the system

### What are we using as building blocks?

In [ ]:
!head ../structures/Na.gro

### `gmx insert-molecules`

- `gmx insert-molecules` is designed to insert molecules into existing structures or simply empty boxes.
- `-ci` &mdash; `.gro` structure to insert
- `-f` &mdash; (optional) `.gro` structure into which to insert
- `-nmol` &mdash; number of molecules to insert
- `-o` &mdash; name of the output `.gro` structure
- `-box` &mdash; define the box size if you do not have any initial structure to insert into (defined by `-f`)
- `-try` &mdash; an attempt to insert the structure could be unsuccessful because of an overlap with already present molecules or/and not enough available free space; therefore, we should specify total number of possible attempts
- `-replace` &mdash; which molecules you can replace upon inserting
- there are other options that we will not cover today; check `gmx insert-molecules -h` or https://manual.gromacs.org/current/onlinehelp/gmx-insert-molecules.html yourself

In [ ]:
!gmx insert-molecules -ci ../structures/Na.gro -o Na.gro -nmol 200 -box 2 2 2 -try 2000

### What did we do?

- We inserted 200 sodium ions into an empty box with a cubic size 2 nm.

In [ ]:
!gmx insert-molecules -ci ../structures/Cl.gro -f Na.gro -o Na_Cl.gro -nmol 200 -try 2000

### What did we do?

- We inserted additionally 200 chloride ions into just created box with sodium cations.

In [ ]:
!gmx editconf -f Na_Cl.gro -o ions.gro -box 6 6 6

### What did we do?

- We changed the box size from 2 nm to 6 nm.
- Now we have a "crystal" (although it's not true, simply a piece) of NaCl in the middle of the box.

<img src='../../images/box_with_ions_before.png' width=600 height=450 alt='box_with_ions_before'>

In [ ]:
!gmx solvate -cp ions.gro -cs ../structures/water_three_site.gro -o ions_in_water.gro -maxsol 6000

### What did we do?

- We solvated NaCl in 6000 water molecules (three-site model).
- You didn't forget about three-site, four-site, etc. models, right?

<img src='../../images/box_with_ions.png' width=600 height=450 alt='box_with_ions'>

### Writing topology

In [ ]:
topology = """; force field parameters
#include "../force_field/charmm36-jul2022.ff/forcefield.itp"
#include "../force_field/charmm36-jul2022.ff/tip3p.itp"
#include "../force_field/charmm36-jul2022.ff/ions.itp"

[ system ]
; name of the system
200 NaCl ion pairs in 6000 waters

[ molecules ]
; compound   #mols
NA           200
CL           200
SOL          6000
"""

with open('topol.top', 'w') as outFile:
    outFile.write(topology)

### Let's check force field files more carefully today
- We know more now.

### Topology of water

In [ ]:
!cat ../force_field/charmm36-jul2022.ff/tip3p.itp

### Topology of ions

In [ ]:
!head -n 50 ../force_field/charmm36-jul2022.ff/ions.itp

### Parameters for nonbonded interactions

- Different atom types have different parameters.
- Atom types used for a specific molecule given in the topology as we showed above.

In [ ]:
!sed -n '51,100p' ../force_field/charmm36-jul2022.ff/ffnonbonded.itp

### Back to simulations

### Minimizing the system
- Recalling the previous week.

In [ ]:
!gmx grompp -p topol.top -c ions_in_water.gro -f ../scripts/minim.mdp -o minim.tpr -po minim_mdout.mdp

In [ ]:
!gmx mdrun -deffnm minim -v

### TASK
- Recalling previous week.
- Checking the potential energy.

### Equilibration and production

In [ ]:
# !gmx grompp -maxwarn 2 -p topol.top -c minim.gro -f ../scripts/eq.mdp -o eq.tpr -po eq_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm eq -v

In [ ]:
# !gmx grompp -p topol.top -c eq.gro -f ../scripts/prod.mdp -o prod.tpr -po prod_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm prod -v

### These simulations are a bit costly
- We will not run these simulations now (commands are commented).
- They were already performed for you.
- Let's copy the trajectories in the current directory.

In [ ]:
!cp ../trajectories/ions/* .

### More discussion about `.mdp` file

In [ ]:
!cat ../scripts/eq.mdp

In [ ]:
Video('../videos/box_with_ions_eq.mp4', width=800, height=600, embed=True)

## Before and after

<img src='../../images/box_with_ions_before.png' width=600 height=450 alt='box_with_ions_before'>
<img src='../../images/box_with_ions_after.png' width=600 height=450 alt='box_with_ions_after'>

### Dissolution of the cluster
- The ionic cluster dissolved within 10-15 ns.
- The time scale force field dependent but multiple studies exist that focus on dissolution (in tandem with experimental techniques).

### Some numerical analysis

### `gmx mindist` 

- `gmx mindist` computes the distance and number of contacts between two atom groups.
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-s` &mdash; (optional) `.tpr` file that has default atom groups, which can be used for the analysis
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-d` &mdash; distance for contacts
- there are other options that we will not cover today; check `gmx mindist -h` or https://manual.gromacs.org/current/onlinehelp/gmx-mindist.html yourself

### Number of contacts
- We will calculate number of contacts of all ions with each other.

In [ ]:
input_command = """Ion
Ion
"""
!echo "$input_command" | gmx mindist -f eq.xtc -s eq.tpr -d 0.5 -od mindist_eq.xvg -on numcont_eq.xvg

In [ ]:
contacts_eq = np.loadtxt('numcont_eq.xvg', comments=['#', '@'])
plt.plot(contacts_eq[:, 0], contacts_eq[:, 1], 'r*')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Number of ion contacts')
plt.show()

### Creating index file
- We should create an index file since it is mandatory for some commands

### `gmx make_ndx` 

- `gmx make_ndx` creates a file with predefined custom groups of atoms.
- `-f` &mdash; input structure, `.gro`
- `-n` &mdash; existing `.ndx` file (if already available)
- `-b` &mdash; output `.ndx` file
- more options at `gmx make_ndx -h` or https://manual.gromacs.org/current/onlinehelp/gmx-make_ndx.html yourself

In [ ]:
input_command = """a NA
a CL
q
"""
!echo "$input_command" | gmx make_ndx -f prod.gro -o index.ndx

### Cluster analysis
- Apparently, we talk about a cluster of ions here, so for this there is a dedicated tool in GROMACS.

### `gmx clustsize` 

- `gmx clustsize` calculates the number and size of clusters in your systems.
- `-f` &mdash; input trajectory, `.xtc` in our case
- `-n` &mdash; (mandatory) `.ndx` file with defined atom groups
- `-s` &mdash; (optional) `.tpr` file
- `-b` &mdash; simulation time to start analysis (useful when you want to analyze only a fragment of your simulation)
- `-e` &mdash; simulation time to end analysis (useful when you want to analyze only a fragment of your simulation)
- `-cut` &mdash; largest distance to be considered in a cluster
- there are other options that we will not cover today; check `gmx clustsize -h` or https://manual.gromacs.org/current/onlinehelp/gmx-clustsize.html yourself

In [ ]:
input_command = """1
"""
!echo "$input_command" | gmx clustsize -f eq.xtc -n index.ndx -cut 0.5 -nc nclust_eq.xvg -mc maxclust_eq.xvg

### Question
- What does 1 in input mean?

### Let us check what we have generated...

In [ ]:
!ls -l

### Generated outputs
- Multiple outputs are generated by default.
- Today we are interested in two which we named explicitly.
- `nclust_eq.xvg` &mdash; number of clusters as the function of time
- `maxclust_eq.xvg` &mdash; maximum size of a cluster as the function of time
- ...

In [ ]:
maxclust_eq = np.loadtxt('maxclust_eq.xvg', comments=['#', '@'])
plt.plot(maxclust_eq[:, 0], maxclust_eq[:, 1], 'r*')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Maximum number of species in a cluster')
plt.show()

### Explain the results

### TASK
- Plot the number of clusters.